In [ ]:
"""
第4部分：RAG问答系统（结合大模型）
"""

from elasticsearch import Elasticsearch
from sentence_transformers import SentenceTransformer
import openai  # 或使用其他LLM API


class MedicalRAGSystem:
    """医疗指南RAG问答系统"""
    
    def __init__(self, es, index_name='medical_guidelines',
                 model_name='paraphrase-multilingual-MiniLM-L12-v2',
                 llm_api_key=None):
        self.es = es
        self.index_name = index_name
        self.embedding_model = SentenceTransformer(model_name)
        
        # 配置LLM（这里以OpenAI为例，可替换为其他）
        if llm_api_key:
            openai.api_key = llm_api_key
    
    # ==================== 1. 检索相关文档 ====================
    def retrieve_relevant_docs(self, question, top_k=5):
        """检索与问题相关的文档"""
        
        # 生成问题向量
        query_vector = self.embedding_model.encode(question)
        
        # 向量搜索
        script_query = {
            "script_score": {
                "query": {"match_all": {}},
                "script": {
                    "source": "cosineSimilarity(params.query_vector, 'vector') + 1.0",
                    "params": {"query_vector": query_vector.tolist()}
                }
            }
        }
        
        query = {
            "size": top_k,
            "query": script_query,
            "_source": {"excludes": ["vector"]}
        }
        
        result = self.es.search(index=self.index_name, body=query)
        
        # 提取文档
        docs = []
        for hit in result['hits']['hits']:
            similarity = hit['_score'] - 1.0
            if similarity > 0.5:  # 相似度阈值
                docs.append({
                    'similarity': similarity,
                    'content': hit['_source']
                })
        
        return docs
    
    # ==================== 2. 构建Prompt ====================
    def build_prompt(self, question, retrieved_docs):
        """构建发送给LLM的Prompt"""
        
        # 系统提示
        system_prompt = """你是一个专业的医疗助手，专门解答偏头痛治疗相关的问题。
请基于提供的医疗指南信息，准确、专业地回答用户的问题。

注意事项：
1. 只使用提供的指南信息，不要编造内容
2. 如果信息不足以回答问题，请明确说明
3. 提供用药建议时，必须注明证据等级和推荐强度
4. 说明禁忌症和不良反应
5. 建议用户咨询专业医生
"""
        
        # 构建上下文
        context = "【医疗指南信息】\n\n"
        for i, doc in enumerate(retrieved_docs, 1):
            content = doc['content']
            intervention = content['intervention']
            evidence = content['evidence']
            safety = content['safety']
            
            context += f"参考{i} (相似度: {doc['similarity']:.2f}):\n"
            context += f"药物: {intervention['name']}\n"
            context += f"分类: {intervention.get('category', '未指定')}\n"
            context += f"治疗阶段: {content['phase']}\n"
            context += f"给药方式: {intervention['route'].get('description', '')}\n"
            context += f"用药频次: {intervention.get('frequency', '')}\n"
            context += f"证据等级: {evidence.get('evidence_level', '')}\n"
            context += f"推荐强度: {evidence.get('recommendation_grade', '')}\n"
            
            if safety.get('contraindications'):
                context += f"禁忌症: {', '.join(safety['contraindications'][:3])}\n"
            
            if safety['adverse_effects'].get('common'):
                context += f"不良反应: {safety['adverse_effects']['common'][:100]}\n"
            
            context += f"来源: {content['source']['guideline']} ({content['source']['year']})\n"
            context += "\n" + "-"*60 + "\n\n"
        
        # 用户问题
        user_prompt = f"【用户问题】\n{question}\n\n请基于上述医疗指南信息回答问题。"
        
        return system_prompt, context + user_prompt
    
    # ==================== 3. 调用LLM生成回答 ====================
    def generate_answer(self, question, use_llm=True):
        """生成回答"""
        
        # 步骤1: 检索相关文档
        print("🔍 正在检索相关医疗指南...")
        retrieved_docs = self.retrieve_relevant_docs(question, top_k=5)
        
        if not retrieved_docs:
            return "抱歉，未找到相关的医疗指南信息。请咨询专业医生。"
        
        print(f"✅ 找到 {len(retrieved_docs)} 条相关信息")
        
        # 步骤2: 构建Prompt
        system_prompt, user_prompt = self.build_prompt(question, retrieved_docs)
        
        # 步骤3: 调用LLM（如果启用）
        if use_llm:
            try:
                response = openai.ChatCompletion.create(
                    model="gpt-3.5-turbo",  # 或gpt-4
                    messages=[
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": user_prompt}
                    ],
                    temperature=0.3,  # 较低温度以确保准确性
                    max_tokens=800
                )
                answer = response.choices[0].message.content
            except Exception as e:
                print(f"LLM调用失败: {e}")
                answer = self._fallback_answer(retrieved_docs)
        else:
            # 不使用LLM，直接返回检索结果摘要
            answer = self._fallback_answer(retrieved_docs)
        
        return answer, retrieved_docs
    
    # ==================== 4. 备用回答（不使用LLM）====================
    def _fallback_answer(self, retrieved_docs):
        """备用回答方案（基于模板）"""
        
        answer = "根据医疗指南，以下是相关治疗建议：\n\n"
        
        for i, doc in enumerate(retrieved_docs[:3], 1):
            content = doc['content']
            intervention = content['intervention']
            
            answer += f"{i}. {intervention['name']}"
            
            if intervention.get('category'):
                answer += f" ({intervention['category']})"
            
            answer += f"\n   治疗阶段: {content['phase']}\n"
            answer += f"   给药方式: {intervention['route'].get('description', '')}\n"
            
            if intervention.get('frequency'):
                answer += f"   用药频次: {intervention['frequency']}\n"
            
            if content['evidence'].get('recommendation_grade'):
                answer += f"   推荐强度: {content['evidence']['recommendation_grade']}\n"
            
            if content['safety'].get('contraindications'):
                answer += f"   禁忌症: {', '.join(content['safety']['contraindications'][:2])}\n"
            
            answer += "\n"
        
        answer += "\n⚠️ 以上信息仅供参考，具体用药请遵医嘱。"
        
        return answer
    
    # ==================== 5. 完整问答流程 ====================
    def ask(self, question, use_llm=False):
        """完整问答流程"""
        
        print(f"\n{'='*80}")
        print(f"问题: {question}")
        print(f"{'='*80}\n")
        
        # 生成回答
        answer, retrieved_docs = self.generate_answer(question, use_llm)
        
        # 打印回答
        print("【回答】")
        print(answer)
        print(f"\n{'='*80}")
        
        # 打印引用来源
        print("\n【参考来源】")
        for i, doc in enumerate(retrieved_docs, 1):
            content = doc['content']
            print(f"{i}. {content['intervention']['name']} - "
                  f"{content['source']['guideline']} ({content['source']['year']})")
        
        print(f"\n{'='*80}\n")
        
        return answer


# ==================== 使用示例 ====================
if __name__ == "__main__":
    # 连接ES
    es = Elasticsearch([{'host': 'localhost', 'port': 9200, 'scheme': 'http'}])
    
    # 创建RAG系统（不使用LLM的简化版本）
    rag = MedicalRAGSystem(es)
    
    # 示例问题
    questions = [
        "偏头痛急性发作时应该吃什么药？",
        "布洛芬治疗偏头痛的剂量是多少？",
        "孕妇偏头痛可以用什么药？",
        "如何预防偏头痛发作？",
        "曲普坦类药物有哪些禁忌症？"
    ]
    
    for question in questions:
        rag.ask(question, use_llm=False)
        print("\n" + "="*100 + "\n")


# ==================== 进阶：使用本地LLM（如使用transformers）====================
class LocalLLMRAG(MedicalRAGSystem):
    """使用本地LLM的RAG系统"""
    
    def __init__(self, es, index_name='medical_guidelines',
                 llm_model='THUDM/chatglm3-6b'):
        super().__init__(es, index_name, llm_api_key=None)
        
        # 加载本地LLM（示例）
        from transformers import AutoTokenizer, AutoModel
        print(f"正在加载本地LLM: {llm_model}...")
        self.tokenizer = AutoTokenizer.from_pretrained(llm_model, trust_remote_code=True)
        self.llm = AutoModel.from_pretrained(llm_model, trust_remote_code=True).half().cuda()
        self.llm = self.llm.eval()
        print("✅ LLM加载完成")
    
    def generate_answer(self, question, use_llm=True):
        """使用本地LLM生成回答"""
        
        # 检索文档
        retrieved_docs = self.retrieve_relevant_docs(question, top_k=5)
        
        if not retrieved_docs:
            return "抱歉，未找到相关信息。", []
        
        # 构建Prompt
        system_prompt, user_prompt = self.build_prompt(question, retrieved_docs)
        
        # 调用本地LLM
        full_prompt = system_prompt + "\n\n" + user_prompt
        response, history = self.llm.chat(self.tokenizer, full_prompt, history=[])
        
        return response, retrieved_docs